# 02 Monza — de la vuelta al perfil de piloto

Tabla larga (piloto × vuelta × zona) → colapso por tipología → mediana por piloto. Ese vector es el perfil que se clusteriza.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from f1_style_clustering.config import load_circuit_config, load_config
from f1_style_clustering.data.loader import enable_cache, load_session
from f1_style_clustering.features.aggregators import (
    CORE_STYLE_FEATURES,
    aggregate_driver_style_features,
    build_feature_matrix,
    build_long_style_table,
    collapse_to_driver_lap_typology,
)
from f1_style_clustering.features.preprocessor import normalize_typology_session_median
from f1_style_clustering.features.zoi import extract_circuit_zones, resolve_selected_corners

In [ ]:
cfg = load_config()
circuits = load_circuit_config()
enable_cache(cfg["general"]["cache_dir"])
session = load_session(2025, "Italian Grand Prix", "R")
selected, source = resolve_selected_corners(session, circuits)
zones = extract_circuit_zones(session, circuit_config=circuits, selected_corners=selected)
print("zones:", list(zones), "source:", source)

Cada fila es un piloto en una vuelta y una zona. Las columnas son las métricas de frenado, vértice y salida.


In [ ]:
long_df = build_long_style_table(session, analysis_zones=zones, circuit_config=circuits)
print(long_df.shape)
long_df.head()

Se resta la mediana de la sesión dentro de cada tipología (slow/medium/fast) y se colapsa a una fila por piloto × vuelta × tipología. Varias curvas del mismo tipo no inflan el vector.


In [ ]:
feature_cols = [c for c in CORE_STYLE_FEATURES if c in long_df.columns]
long_rel = normalize_typology_session_median(long_df, feature_cols=feature_cols)
lap_typ = collapse_to_driver_lap_typology(long_rel, feature_cols=feature_cols)
print("long", long_df.shape, "→ lap×typology", lap_typ.shape)
lap_typ.head()

Mediana entre vueltas: un vector por piloto. `build_feature_matrix` hace estos tres pasos juntos.


In [ ]:
profiles = aggregate_driver_style_features(lap_typ, method="median", feature_cols=feature_cols)
print(profiles.shape)
profiles[feature_cols].head()

In [ ]:
df_raw = build_feature_matrix(session, analysis_zones=zones, circuit_config=circuits, aggregation="median")
df_raw.head()